# APTOS 2019 Blindness Detection — Exploratory Data Analysis

| Label | Mức độ |
|-------|--------|
| 0 | No DR (Không bệnh) |
| 1 | Mild (Nhẹ) |
| 2 | Moderate (Trung bình) |
| 3 | Severe (Nặng) |
| 4 | Proliferative DR (Tăng sinh) |

## 1. Loading libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

BASE_DIR  = '/users/PGS0404/atran16/Desktop/diabeticRetinopathyGrading/APTOS_2019_datasets'
TRAIN_IMG = os.path.join(BASE_DIR, 'train_images')
TEST_IMG  = os.path.join(BASE_DIR, 'test_images')
SAVE_DIR  = '/users/PGS0404/atran16/Desktop/diabeticRetinopathyGrading'

CLASS_NAMES = {0: 'No DR', 1: 'Mild', 2: 'Moderate', 3: 'Severe', 4: 'Proliferative DR'}
PALETTE     = ['#2ecc71', '#3498db', '#f39c12', '#e67e22', '#e74c3c']

train_df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))

print(f'Train set : {len(train_df):,} samples')
print(f'Test  set : {len(test_df):,} samples')
print('\nTrain head:')
display(train_df.head())
print('\nTest head (no labels):')
display(test_df.head())

## 2. Pie chart distribution — Train Set

In [ ]:
counts = train_df['diagnosis'].value_counts().sort_index()
total  = len(train_df)
xlabels = [f'{CLASS_NAMES[i]}\n(Class {i})' for i in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Phân phối nhãn — Train Set (3,662 ảnh)', fontsize=14, fontweight='bold')

# Pie chart
wedges, texts, autotexts = axes[0].pie(
    counts.values, labels=xlabels, colors=PALETTE,
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for t in autotexts:
    t.set_fontsize(9)
axes[0].set_title('Tỷ lệ phần trăm từng class', fontsize=12)

# Donut chart (test set)
axes[1].pie(
    [len(test_df)], labels=[f'Test Set\n{len(test_df):,} ảnh\n(không có nhãn)'],
    colors=['#95a5a6'], startangle=90,
    wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2)
)
axes[1].set_title('Test / Validation Set', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_02_pie_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

# Summary table
df_summary = pd.DataFrame({
    'Class': [CLASS_NAMES[i] for i in counts.index],
    'Số lượng': counts.values,
    'Tỷ lệ (%)': (counts.values / total * 100).round(2)
})
display(df_summary)

## 3. Column chart distribution — Train Set

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân phối số lượng mẫu — Train Set', fontsize=14, fontweight='bold')

# Bar chart
bars = axes[0].bar(xlabels, counts.values, color=PALETTE, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                 f'{val:,}\n({val/total*100:.1f}%)', ha='center', va='bottom', fontsize=9)
axes[0].set_title('Số lượng ảnh theo class')
axes[0].set_ylabel('Số lượng ảnh')
axes[0].set_ylim(0, counts.max() * 1.22)

# Horizontal bar — imbalance
norm = counts / counts.max()
hbars = axes[1].barh([CLASS_NAMES[i] for i in counts.index], norm.values,
                      color=PALETTE, edgecolor='white', linewidth=1.5)
for bar, v, raw in zip(hbars, norm.values, counts.values):
    axes[1].text(v + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{raw:,}  ({v*100:.1f}%)', va='center', fontsize=9)
axes[1].axvline(1.0, color='gray', linestyle='--', linewidth=1, label='Class đa số')
axes[1].set_xlim(0, 1.38)
axes[1].set_xlabel('Tỷ lệ so với class lớn nhất')
axes[1].set_title(f'Mất cân bằng class  (ratio = {counts.max()/counts.min():.1f}x)')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_03_bar_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Ảnh mẫu theo từng class — Train Set

In [ ]:
N_COLS = 5
fig, axes = plt.subplots(5, N_COLS, figsize=(15, 14))
fig.suptitle('Ảnh mẫu theo từng mức độ bệnh — Train Set', fontsize=14, fontweight='bold')

for cls in range(5):
    ids = train_df[train_df['diagnosis'] == cls]['id_code'].sample(
        min(N_COLS, (train_df['diagnosis'] == cls).sum()), random_state=42
    ).values
    for col, img_id in enumerate(ids):
        img = Image.open(os.path.join(TRAIN_IMG, f'{img_id}.png')).resize((224, 224))
        axes[cls][col].imshow(img)
        axes[cls][col].axis('off')
    axes[cls][0].set_ylabel(f'Class {cls}\n{CLASS_NAMES[cls]}',
                             fontsize=10, fontweight='bold', color=PALETTE[cls],
                             rotation=90, labelpad=5)
    axes[cls][0].yaxis.set_visible(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_04_train_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Ảnh mẫu ngẫu nhiên — Test (Validation) Set

In [ ]:
samples = test_df['id_code'].sample(20, random_state=42).values

fig, axes = plt.subplots(4, 5, figsize=(15, 12))
fig.suptitle(f'Ảnh mẫu ngẫu nhiên — Test/Validation Set ({len(test_df):,} ảnh, không có nhãn)',
             fontsize=13, fontweight='bold')

for ax, img_id in zip(axes.flatten(), samples):
    img = Image.open(os.path.join(TEST_IMG, f'{img_id}.png')).resize((224, 224))
    ax.imshow(img)
    ax.set_title(img_id[:14], fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_05_test_samples.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Phân phối kích thước ảnh (Train vs Test)

In [ ]:
def get_sizes(img_dir, ids, n=300):
    ws, hs = [], []
    for img_id in np.random.choice(ids, min(n, len(ids)), replace=False):
        try:
            w, h = Image.open(os.path.join(img_dir, f'{img_id}.png')).size
            ws.append(w); hs.append(h)
        except: pass
    return ws, hs

print('Đang đọc kích thước ảnh (sample 300/set)...')
tr_w, tr_h = get_sizes(TRAIN_IMG, train_df['id_code'].values)
te_w, te_h = get_sizes(TEST_IMG,  test_df['id_code'].values)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Phân phối kích thước ảnh', fontsize=14, fontweight='bold')

configs = [
    (tr_w, 'Train — Chiều rộng (px)', '#3498db'),
    (tr_h, 'Train — Chiều cao (px)',  '#3498db'),
    (te_w, 'Test  — Chiều rộng (px)', '#e67e22'),
    (te_h, 'Test  — Chiều cao (px)',  '#e67e22'),
]
for ax, (data, title, color) in zip(axes.flatten(), configs):
    ax.hist(data, bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(data),   color='red',   linestyle='--', lw=1.5, label=f'Mean: {np.mean(data):.0f}px')
    ax.axvline(np.median(data), color='black', linestyle=':',  lw=1.5, label=f'Median: {np.median(data):.0f}px')
    ax.set_title(title); ax.set_xlabel('Pixels'); ax.set_ylabel('Số ảnh')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_06_image_sizes.png'), dpi=150, bbox_inches='tight')
plt.show()

for label, ws, hs in [('Train', tr_w, tr_h), ('Test ', te_w, te_h)]:
    print(f'{label} | W: min={min(ws):4d}  max={max(ws):4d}  mean={np.mean(ws):.0f}  '
          f'| H: min={min(hs):4d}  max={max(hs):4d}  mean={np.mean(hs):.0f}')

## 7. Phân phối độ sáng (Brightness) — Train theo class & Train vs Test

In [ ]:
def get_brightness(img_dir, ids, n=80):
    vals = []
    for img_id in np.random.choice(ids, min(n, len(ids)), replace=False):
        img = cv2.imread(os.path.join(img_dir, f'{img_id}.png'))
        if img is not None:
            vals.append(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).mean())
    return vals

print('Đang tính độ sáng (80 ảnh/class, 200 ảnh test)...')
bright_cls  = {c: get_brightness(TRAIN_IMG, train_df[train_df['diagnosis']==c]['id_code'].values)
               for c in range(5)}
bright_test = get_brightness(TEST_IMG, test_df['id_code'].values, n=200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phân phối độ sáng ảnh (Mean Pixel Intensity)', fontsize=13, fontweight='bold')

# Violin per class
parts = axes[0].violinplot([bright_cls[c] for c in range(5)],
                            positions=range(5), showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(PALETTE[i]); pc.set_alpha(0.75)
axes[0].set_xticks(range(5))
axes[0].set_xticklabels([f'Class {c}\n{CLASS_NAMES[c]}' for c in range(5)], fontsize=8)
axes[0].set_ylabel('Mean pixel intensity')
axes[0].set_title('Train — Độ sáng theo class')

# Train vs Test
all_train = [v for lst in bright_cls.values() for v in lst]
axes[1].hist(all_train,   bins=30, alpha=0.6, color='#3498db',
             label=f'Train (n={len(all_train)})', edgecolor='white')
axes[1].hist(bright_test, bins=30, alpha=0.6, color='#e67e22',
             label=f'Test  (n={len(bright_test)})', edgecolor='white')
axes[1].axvline(np.mean(all_train),   color='#2980b9', linestyle='--', lw=1.5,
                label=f'Train mean: {np.mean(all_train):.1f}')
axes[1].axvline(np.mean(bright_test), color='#d35400', linestyle='--', lw=1.5,
                label=f'Test  mean: {np.mean(bright_test):.1f}')
axes[1].set_xlabel('Mean pixel intensity'); axes[1].set_ylabel('Số ảnh')
axes[1].set_title('So sánh độ sáng Train vs Test'); axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_07_brightness.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Ảnh trung bình (Mean Image) & Độ lệch chuẩn theo class

In [ ]:
SIZE  = (256, 256)
N_PER = 50

print('Đang tính mean/std images (50 ảnh/class)...')
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Ảnh trung bình & Độ lệch chuẩn theo class (n=50/class)',
             fontsize=13, fontweight='bold')

for cls in range(5):
    ids = train_df[train_df['diagnosis'] == cls]['id_code'].sample(
        min(N_PER, (train_df['diagnosis'] == cls).sum()), random_state=42
    ).values
    stack = []
    for img_id in ids:
        img = cv2.imread(os.path.join(TRAIN_IMG, f'{img_id}.png'))
        if img is not None:
            img = cv2.cvtColor(cv2.resize(img, SIZE), cv2.COLOR_BGR2RGB)
            stack.append(img.astype(np.float32))
    if stack:
        arr      = np.stack(stack)
        mean_img = arr.mean(axis=0).astype(np.uint8)
        std_max  = arr.std(axis=0).max()
        std_img  = (arr.std(axis=0) / (std_max + 1e-6) * 255).astype(np.uint8)

        axes[0][cls].imshow(mean_img)
        axes[0][cls].set_title(f'Class {cls} — {CLASS_NAMES[cls]}\nMean Image',
                                fontsize=9, color=PALETTE[cls], fontweight='bold')
        axes[0][cls].axis('off')

        axes[1][cls].imshow(std_img)
        axes[1][cls].set_title('Std Image\n(vùng biến động)', fontsize=9)
        axes[1][cls].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_08_mean_std_images.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Phân phối kênh màu RGB theo class

In [ ]:
def sample_rgb(img_dir, ids, n=25):
    r, g, b = [], [], []
    for img_id in np.random.choice(ids, min(n, len(ids)), replace=False):
        img = cv2.imread(os.path.join(img_dir, f'{img_id}.png'))
        if img is not None:
            img = cv2.cvtColor(cv2.resize(img, (128, 128)), cv2.COLOR_BGR2RGB)
            r.extend(img[:,:,0].flatten().tolist())
            g.extend(img[:,:,1].flatten().tolist())
            b.extend(img[:,:,2].flatten().tolist())
    return r, g, b

print('Đang lấy mẫu kênh màu (25 ảnh/class)...')
fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Phân phối kênh màu RGB theo class — Train Set',
             fontsize=13, fontweight='bold')

for cls in range(5):
    ids = train_df[train_df['diagnosis'] == cls]['id_code'].values
    r, g, b = sample_rgb(TRAIN_IMG, ids)
    axes[cls].hist(r, bins=50, alpha=0.55, color='red',   label='R', density=True)
    axes[cls].hist(g, bins=50, alpha=0.55, color='green', label='G', density=True)
    axes[cls].hist(b, bins=50, alpha=0.55, color='blue',  label='B', density=True)
    axes[cls].set_title(f'Class {cls}\n{CLASS_NAMES[cls]}',
                        fontsize=9, color=PALETTE[cls], fontweight='bold')
    axes[cls].set_xlabel('Pixel value')
    if cls == 0:
        axes[cls].set_ylabel('Density')
        axes[cls].legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'plot_09_rgb_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 10. Tổng hợp thống kê

In [ ]:
counts = train_df['diagnosis'].value_counts().sort_index()

print('=' * 58)
print('           TỔNG KẾT EDA — APTOS 2019 DATASET')
print('=' * 58)
print(f'  Train set : {len(train_df):,} ảnh  (có nhãn 0–4)')
print(f'  Test  set : {len(test_df):,} ảnh  (không có nhãn — competition blind test)')
print('\n  Phân phối nhãn Train:')
for cls, cnt in counts.items():
    bar = '█' * int(cnt / counts.max() * 28)
    print(f'    Class {cls}  {CLASS_NAMES[cls]:<18}: {bar:<29} {cnt:4d} ({cnt/len(train_df)*100:.1f}%)')
print(f'\n  Imbalance ratio   : {counts.max()/counts.min():.1f}x')
print(f'  Class đa số       : {CLASS_NAMES[counts.idxmax()]}  →  {counts.max()} mẫu')
print(f'  Class thiểu số    : {CLASS_NAMES[counts.idxmin()]}  →  {counts.min()} mẫu')
print('\n  Lưu ý khi huấn luyện:')
print('    - Ảnh kích thước không đồng nhất  → cần resize / padding')
print('    - Độ sáng chênh lệch lớn          → nên dùng CLAHE hoặc normalization')
print('    - Mất cân bằng class nghiêm trọng → weighted loss hoặc oversampling')
print('    - Test set không có nhãn          → dùng cross-validation để đánh giá')
print('\n  Các biểu đồ đã lưu tại:', SAVE_DIR)